In [ ]:
import numpy as np
import heapq
import pandas as pd
from scipy.stats import skewnorm
import os

SIM_TIME=45*24; DRIVER_RATE=4.74; RIDER_RATE=34.6; PATIENCE_RATE=5
SPEED=20; AREA_SIZE=20; BASE_FARE=3; FARE_PER_MILE=2
PETROL_COST_PER_MILE=0.20; CVAR_ALPHA=0.05

OUTPUT_DIR="/mnt/user-data/outputs/lambda_sweep_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

HUBS=[np.array([5.43,14.86]),np.array([12.84,14.10]),np.array([6.78,7.60])]

def nearest_hub_dist(loc):
    return min(np.linalg.norm(loc-h) for h in HUBS)
def driver_initial_location():
    return np.array([np.clip(np.random.normal(9.97,4.36),0,20),
                     np.clip(np.random.normal(11.51,4.34),0,20)])
def rider_pickup_location():
    return np.array([np.clip(skewnorm.rvs(a=1.80,loc=4.18,scale=5.97),0,20),
                     np.clip(skewnorm.rvs(a=-2.49,loc=17.05,scale=6.31),0,20)])
def rider_dropoff_location():
    return np.array([np.clip(skewnorm.rvs(a=-1.65,loc=15.51,scale=6.24),0,20),
                     np.clip(skewnorm.rvs(a=-14.26,loc=19.50,scale=7.50),0,20)])

class Driver:
    def __init__(self,did,loc,ou):
        self.id=did;self.location=loc;self.online_until=ou
        self.earnings=0;self.status='idle';self.busy_time=0
class Rider:
    def __init__(self,rid,o,d,arr,ab):
        self.id=rid;self.origin=o;self.destination=d
        self.arrival_time=arr;self.abandon_time=ab;self.matched=False

class Simulation:
    def __init__(self,lam=0,variant='baseline'):
        self.lam=lam;self.variant=variant
        self.current_time=0;self.event_list=[]
        self.idle_drivers={};self.busy_drivers={};self.waiting_riders={}
        self.driver_counter=0;self.rider_counter=0
        self.total_riders=0;self.abandoned_riders=0
        self.total_queue_wait=0;self.total_true_wait=0;self.driver_log=[]

    def exp_time(self,r): return np.random.exponential(1/r)
    def dist(self,a,b): return np.linalg.norm(np.array(a)-np.array(b))
    def ttime(self,d): mu=d/SPEED; return np.random.uniform(0.8*mu,1.2*mu)
    def schedule(self,t,e,d=None): heapq.heappush(self.event_list,(t,e,d))

    def select_rider(self,driver):
        def score(rid):
            r=self.waiting_riders[rid]
            d2p=self.dist(driver.location,r.origin)
            if self.variant=='baseline': return d2p
            elif self.variant=='3a': return d2p+self.lam*nearest_hub_dist(r.origin)
            elif self.variant=='3b': return d2p+self.lam*nearest_hub_dist(r.destination)
        return min(self.waiting_riders,key=score)

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders: return
        did=next(iter(self.idle_drivers)); driver=self.idle_drivers[did]
        rid=self.select_rider(driver)
        rider=self.waiting_riders[rid]; rider.matched=True
        pd=self.dist(driver.location,rider.origin)
        td=self.dist(rider.origin,rider.destination)
        pt=self.ttime(pd); tt2=self.ttime(td); tt=pt+tt2
        qw=self.current_time-rider.arrival_time
        self.total_queue_wait+=qw; self.total_true_wait+=qw+pt
        earn=(BASE_FARE+FARE_PER_MILE*td)-PETROL_COST_PER_MILE*(pd+td)
        driver.status='busy'; driver.busy_time+=tt
        self.busy_drivers[did]=driver
        del self.idle_drivers[did]; del self.waiting_riders[rid]
        self.schedule(self.current_time+tt,'DROPOFF',(did,rider.destination,earn))

    def handle_driver_arrival(self):
        self.driver_counter+=1; did=self.driver_counter
        loc=driver_initial_location(); ou=self.current_time+np.random.uniform(6,8)
        self.idle_drivers[did]=Driver(did,loc,ou)
        self.driver_log.append({'driver_id':did,'arrival_time':self.current_time,
            'online_until':ou,'earnings':0,'busy_time':0})
        self.schedule(ou,'DRIVER_OFFLINE',did)
        self.schedule(self.current_time+self.exp_time(DRIVER_RATE),'DRIVER_ARRIVAL')
        self.try_match()

    def handle_rider_arrival(self):
        self.rider_counter+=1; rid=self.rider_counter; self.total_riders+=1
        o=rider_pickup_location(); d=rider_dropoff_location()
        ab=self.current_time+self.exp_time(PATIENCE_RATE)
        self.waiting_riders[rid]=Rider(rid,o,d,self.current_time,ab)
        self.schedule(ab,'RIDER_ABANDON',rid)
        self.schedule(self.current_time+self.exp_time(RIDER_RATE),'RIDER_ARRIVAL')
        self.try_match()

    def handle_dropoff(self,data):
        did,nloc,earn=data; driver=self.busy_drivers[did]
        driver.earnings+=earn; driver.location=nloc; driver.status='idle'
        for d in self.driver_log:
            if d['driver_id']==did: d['earnings']+=earn; d['busy_time']=driver.busy_time; break
        del self.busy_drivers[did]
        if self.current_time<driver.online_until: self.idle_drivers[did]=driver
        self.try_match()

    def run(self):
        self.schedule(0,'DRIVER_ARRIVAL'); self.schedule(0,'RIDER_ARRIVAL')
        while self.event_list and self.current_time<SIM_TIME:
            self.current_time,et,data=heapq.heappop(self.event_list)
            if et=='DRIVER_ARRIVAL': self.handle_driver_arrival()
            elif et=='RIDER_ARRIVAL': self.handle_rider_arrival()
            elif et=='DROPOFF': self.handle_dropoff(data)
            elif et=='RIDER_ABANDON':
                if data in self.waiting_riders:
                    self.abandoned_riders+=1; del self.waiting_riders[data]
            elif et=='DRIVER_OFFLINE': self.idle_drivers.pop(data,None)
        return self.collect_results()

    def collect_results(self):
        matched=self.total_riders-self.abandoned_riders
        df=pd.DataFrame(self.driver_log)
        df['online_duration']=df['online_until']-df['arrival_time']
        df['busy_time_capped']=df[['busy_time','online_duration']].min(axis=1)
        df['idle_time']=df['online_duration']-df['busy_time_capped']
        p=df['earnings'].values; N=len(p); mp=p.mean()
        vp=np.sum((p-mp)**2)/(N-1) if N>1 else 0
        ps=np.sort(p)
        cvar=np.mean(ps[:max(1,int(np.floor(CVAR_ALPHA*N)))])
        cvar_med=np.mean(ps[:max(1,int(np.floor(0.5*N)))])
        return {
            'abandonment_rate':self.abandoned_riders/self.total_riders,
            'avg_queue_wait':self.total_queue_wait/matched if matched>0 else 0,
            'avg_true_wait':self.total_true_wait/matched if matched>0 else 0,
            'matched_riders':matched,'total_riders':self.total_riders,
            'avg_earnings':mp,'total_earnings':df['earnings'].sum(),
            'avg_profit_per_hr':df['earnings'].sum()/df['online_duration'].sum(),
            'avg_idle_time':df['idle_time'].mean(),
            'variance':vp,'fairness_cv':np.sqrt(vp)/mp if mp>0 else 0,
            'cvar':cvar,'delta_cvar_internal':cvar-cvar_med,
            '_df':df,
        }

def fairness_metrics(pre,post):
    return {
        'delta_profit':post['avg_earnings']-pre['avg_earnings'],
        'fairness_effect':1-post['variance']/pre['variance'] if pre['variance']>0 else 0,
        'delta_cvar':post['cvar']-pre['cvar'],
    }

# ─────────────────────────────────────────
# RUN — use smaller lambda range for speed
# ─────────────────────────────────────────

lam_values=[round(x,1) for x in np.arange(0.5,5.1,0.5)]  # 0.5,1.0,...,5.0
print(f"Lambda values to test: {lam_values}")

# Baseline
print("\nRunning Baseline...")
np.random.seed(42)
res_base=Simulation(lam=0,variant='baseline').run()
res_base['_df'].to_csv(f"{OUTPUT_DIR}/data_baseline.csv",index=False)
print(f"  Abandon:{res_base['abandonment_rate']*100:.2f}%  Earn:£{res_base['avg_earnings']:.2f}")

all_rows=[{'variant':'baseline','lambda':0.0,
    **{k:v for k,v in res_base.items() if not k.startswith('_')},
    'fairness_effect':0.0,'delta_cvar_cross':0.0,'delta_profit':0.0}]

for variant in ['3a','3b']:
    print(f"\nRunning {variant} ({'pickup' if variant=='3a' else 'dropoff'}→hub)...")
    print(f"  {'λ':>4} {'Abandon%':>9} {'Queue(min)':>11} {'True(min)':>10} {'AvgEarn£':>9} {'TotalEarn£':>12} {'FairnessEff':>12} {'ΔCVaR£':>9}")
    print("  "+"-"*80)
    for lam in lam_values:
        np.random.seed(42)
        res=Simulation(lam=lam,variant=variant).run()
        fm=fairness_metrics(res_base,res)
        label=f"{variant}_lam{lam:.1f}"
        res['_df'].to_csv(f"{OUTPUT_DIR}/data_{label}.csv",index=False)
        all_rows.append({'variant':variant,'lambda':lam,
            **{k:v for k,v in res.items() if not k.startswith('_')},
            'fairness_effect':fm['fairness_effect'],
            'delta_cvar_cross':fm['delta_cvar'],
            'delta_profit':fm['delta_profit']})
        print(f"  {lam:>4.1f} {res['abandonment_rate']*100:>8.2f}% "
              f"{res['avg_queue_wait']*60:>11.3f} {res['avg_true_wait']*60:>10.3f} "
              f"{res['avg_earnings']:>9.2f} {res['total_earnings']:>12.2f} "
              f"{fm['fairness_effect']:>+12.4f} {fm['delta_cvar']:>+9.2f}")

summary_df=pd.DataFrame([{k:v for k,v in r.items() if k!='_df'} for r in all_rows])
summary_df.to_csv(f"{OUTPUT_DIR}/summary_lambda_sweep.csv",index=False)

# Best configs
print("\n=== BEST CONFIG PER METRIC ===")
non_base=summary_df[summary_df['variant']!='baseline']
for metric,label,ascending in [
    ('abandonment_rate','Lowest Abandonment',True),
    ('avg_earnings','Highest Avg Earnings',False),
    ('fairness_effect','Best Fairness Effect',False),
    ('delta_cvar_cross','Best ΔCVaR',False),
]:
    row=non_base.loc[non_base[metric].idxmin() if ascending else non_base[metric].idxmax()]
    print(f"  {label}: variant={row['variant']}  λ={row['lambda']:.1f}  "
          f"abandon={row['abandonment_rate']*100:.2f}%  earn=£{row['avg_earnings']:.2f}  "
          f"FE={row['fairness_effect']:+.4f}  ΔCVaR={row['delta_cvar_cross']:+.2f}")

print(f"\nSaved to: {OUTPUT_DIR}/")